# 03 — Closed-loop comparison: PID vs LQR

Simulate the mass-spring-damper under PID and LQR control, then compare metrics.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from control_lab.models.lti import LTIModel
from control_lab.design.pid import PIDController
from control_lab.design.lqr import LQRController, lqr_continuous
from control_lab.sim.backend_control import ControlBackend
from control_lab.sim.common import compute_metrics
from control_lab.utils.plotting import plot_trajectories

%matplotlib inline

## MSD + PID closed loop

In [ ]:
model = LTIModel.mass_spring_damper(m=1.0, k=1.0, c=0.5)
backend = ControlBackend()
x0 = np.array([0.0, 0.0])
r_func = lambda t: np.array([1.0])

pid = PIDController(kp=20.0, ki=5.0, kd=3.0, dt=0.01, u_min=-100.0, u_max=100.0)
result_pid = backend.simulate(model, pid, x0, r_func, (0.0, 10.0), 0.01)
metrics_pid = compute_metrics(result_pid, r_final=1.0)
print('PID metrics:', metrics_pid)

## MSD + LQR closed loop

In [ ]:
Q = np.diag([100.0, 1.0])
R = np.array([[0.1]])
K, _, _ = lqr_continuous(model.A, model.B, Q, R)

# LQR regulates to zero; shift reference by feeding forward equilibrium state
x_ref = np.array([1.0, 0.0])  # desired state
r_func_lqr = lambda t: x_ref

lqr_ctrl = LQRController(K)
result_lqr = backend.simulate(model, lqr_ctrl, x0, r_func_lqr, (0.0, 10.0), 0.01)
metrics_lqr = compute_metrics(result_lqr, r_final=1.0)
print('LQR metrics:', metrics_lqr)

## Compare trajectories and metrics

In [ ]:
fig = plot_trajectories(
    {'PID': result_pid, 'LQR': result_lqr},
    title='MSD Closed-Loop: PID vs LQR',
)
plt.axhline(1.0, color='r', linestyle='--', label='Reference')
plt.legend()
plt.show()

import polars as pl
comparison = pl.DataFrame({
    'controller': ['PID', 'LQR'],
    'overshoot_%': [metrics_pid['overshoot'], metrics_lqr['overshoot']],
    'settling_time_s': [metrics_pid['settling_time'], metrics_lqr['settling_time']],
    'iae': [metrics_pid['iae'], metrics_lqr['iae']],
    'ise': [metrics_pid['ise'], metrics_lqr['ise']],
})
print(comparison)